# 01 — Data Audit

Goal: inspect the original Kaggle `listings.csv` without modifying it. The audit establishes grain, integrity, missingness, cardinality and deterministic parsing feasibility before cleaning.

In [1]:
from pathlib import Path
import hashlib
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "listings.csv"
df = pd.read_csv(RAW_PATH, low_memory=False)
df.shape

(3585, 95)

## Grain and uniqueness

The expected grain is one row per Airbnb listing. `id` is tested as the listing identifier.

In [2]:
print("Rows:", len(df))
print("Columns:", df.shape[1])
print("Unique listing IDs:", df["id"].nunique())
print("Duplicate listing IDs:", df["id"].duplicated().sum())
print("Fully duplicated rows:", df.duplicated().sum())

Rows: 3585
Columns: 95
Unique listing IDs: 3585
Duplicate listing IDs: 0
Fully duplicated rows: 0


## Raw-file integrity

In [3]:
sha256 = hashlib.sha256(RAW_PATH.read_bytes()).hexdigest()
print("SHA-256:", sha256)
print("last_scraped values:")
print(df["last_scraped"].value_counts(dropna=False))

SHA-256: 9c2fa5db20b0d1018cb558750aeb276599ea877e1e0c1ae315766a032d78881f
last_scraped values:
last_scraped
2016-09-07    3585
Name: count, dtype: int64


## Schema

In [4]:
schema = pd.DataFrame({"dtype": df.dtypes.astype(str), "missing": df.isna().sum(), "unique": df.nunique(dropna=True)})
schema

,dtype,missing,unique
id,int64,0,3585
listing_url,object,0,3585
scrape_id,int64,0,1
last_scraped,object,0,1
name,object,0,3504
...,...,...,...
cancellation_policy,object,0,4
require_guest_profile_picture,object,0,2
require_guest_phone_verification,object,0,2
calculated_host_listings_count,int64,0,25


## Highest missingness

In [5]:
missing = (df.isna().mean().mul(100).sort_values(ascending=False).to_frame("missing_pct"))
missing.head(20)

,missing_pct
jurisdiction_names,100.00
license,100.00
neighbourhood_group_cleansed,100.00
has_availability,100.00
square_feet,98.44
monthly_price,75.23
weekly_price,75.12
security_deposit,62.57
notes,55.09
interaction,43.35


## Completely empty fields

In [6]:
all_null = [c for c in df.columns if df[c].isna().all()]
all_null

['neighbourhood_group_cleansed',
 'has_availability',
 'license',
 'jurisdiction_names']

## Dashboard-friendly category cardinality

In [7]:
for col in ["room_type", "cancellation_policy", "property_type", "neighbourhood_cleansed", "bed_type"]:
    print(f"{col}: {df[col].nunique(dropna=True)} unique values")
    print(df[col].value_counts(dropna=False).head(30), "\n")

room_type: 3 unique values
room_type
Entire home/apt    2127
Private room       1378
Shared room          80
Name: count, dtype: int64 

cancellation_policy: 4 unique values
cancellation_policy
strict             1582
flexible            999
moderate            919
super_strict_30      85
Name: count, dtype: int64 

property_type: 13 unique values
property_type
Apartment          2612
House               562
Condominium         231
Townhouse            54
Bed & Breakfast      41
Loft                 39
Other                17
Boat                 12
Villa                 6
Entire Floor          4
NaN                   3
Dorm                  2
Guesthouse            1
Camper/RV             1
Name: count, dtype: int64 

neighbourhood_cleansed: 25 unique values
neighbourhood_cleansed
Jamaica Plain              343
South End                  326
Back Bay                   302
Fenway                     290
Dorchester                 269
Allston                    260
Beacon Hill           

## Numeric conversion checks

These checks do not modify the raw dataframe. They verify whether currency and percentage strings can be converted without losing non-null source values.

In [8]:
def parse_currency(series):
    return pd.to_numeric(series.astype("string").str.replace("$", "", regex=False).str.replace(",", "", regex=False), errors="coerce")

def parse_pct(series):
    return pd.to_numeric(series.astype("string").str.replace("%", "", regex=False), errors="coerce")

for col in ["price", "cleaning_fee", "extra_people"]:
    parsed = parse_currency(df[col])
    failed = ((df[col].notna()) & parsed.isna()).sum()
    print(col, "non-null source:", df[col].notna().sum(), "parse failures:", failed, "range:", (parsed.min(), parsed.max()))

for col in ["host_response_rate", "host_acceptance_rate"]:
    parsed = parse_pct(df[col])
    failed = ((df[col].notna()) & parsed.isna()).sum()
    print(col, "non-null source:", df[col].notna().sum(), "parse failures:", failed)

price non-null source: 3585 parse failures: 0 range: (np.float64(10.0), np.float64(4000.0))
cleaning_fee non-null source: 2478 parse failures: 0 range: (np.float64(5.0), np.float64(300.0))
extra_people non-null source: 3585 parse failures: 0 range: (np.float64(0.0), np.float64(200.0))
host_response_rate non-null source: 3114 parse failures: 0


host_acceptance_rate non-null source: 3114 parse failures: 0


## Audit decision

- No duplicated listing IDs.
- No fully duplicated rows.
- No row deletion is justified.
- Missing values will remain missing.
- Extreme prices remain because this file alone does not prove they are invalid.
- Cleaning will select analysis-relevant columns and perform deterministic type conversions only.